In [1]:
import pandas as pd

roll_number = "1024160124"
categories_map = ["billing", "account", "general"]

d1 = int(roll_number[-2])
d2 = int(roll_number[-1])

cat1 = categories_map[d1 % 3]
cat2 = categories_map[d2 % 3]

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
]

personalized_entries = [
    {"question": "how do i update my registered mobile number", "answer": "Go to Profile > Edit Mobile Number.", "keywords": "update mobile phone", "category": cat1},
    {"question": "how can i change my email address", "answer": "Navigate to Account Settings > Update Email.", "keywords": "change email mail", "category": cat2}
]

df = pd.DataFrame(fixed_entries + personalized_entries)
df

,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my registered mobile number,Go to Profile > Edit Mobile Number.,update mobile phone,general
5,how can i change my email address,Navigate to Account Settings > Update Email.,change email mail,account


In [2]:
def score_faq(query, df):
    query_words = set(query.lower().split())
    
    def compute_score(row):
        kw = set(row['keywords'].lower().split())
        q = set(row['question'].lower().split())
        return len(query_words.intersection(kw)) * 2 + len(query_words.intersection(q))

    df_scored = df.copy()
    df_scored['score'] = df_scored.apply(compute_score, axis=1)
    return df_scored[df_scored['score'] > 0].sort_values(by='score', ascending=False)

score_faq("how to reset password fee", df)

,question,answer,keywords,category,score
1,how to reset password,Go to Settings > Reset Password.,password reset login,account,8
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,4
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing,3
4,how do i update my registered mobile number,Go to Profile > Edit Mobile Number.,update mobile phone,general,1
5,how can i change my email address,Navigate to Account Settings > Update Email.,change email mail,account,1


In [3]:
def same_category(category_name, df):
    return df[df['category'] == category_name]['question'].tolist()

print(same_category(cat1, df))

['what are your working hours', 'how do i update my registered mobile number']


In [4]:
new_keyword = input("Enter a new keyword to add: ")
target_idx = 4

df.at[target_idx, 'keywords'] = df.at[target_idx, 'keywords'] + " " + new_keyword.strip()
df.to_csv(f"{roll_number}_faq_data.csv", index=False)

In [5]:
category_counts = df.groupby('category').size()
print(category_counts)

category
account    2
billing    2
general    2
dtype: int64


In [6]:
def score_faq_with_ties(query, df):
    query_words = set(query.lower().split())
    
    def compute_score(row):
        kw = set(row['keywords'].lower().split())
        q = set(row['question'].lower().split())
        return len(query_words.intersection(kw)) * 2 + len(query_words.intersection(q))

    df_scored = df.copy()
    df_scored['score'] = df_scored.apply(compute_score, axis=1)
    df_matched = df_scored[df_scored['score'] > 0]
    
    if df_matched.empty:
        return df_matched
    
    max_score = df_matched['score'].max()
    top_matches = df_matched[df_matched['score'] == max_score]
    return top_matches

print(score_faq_with_ties("fee", df))
print(score_faq_with_ties("working hours timing", df))

                 question                                      answer  \
0  what is the annual fee                   The annual fee is Rs 500.   
3   how can i pay the fee  You can pay via UPI, card, or net banking.   

                keywords category  score  
0  fee cost price charge  billing      3  
3    pay payment upi fee  billing      3  
                      question                     answer  \
2  what are your working hours  We are open 9 AM to 5 PM.   

                 keywords category  score  
2  hours timing open time  general      6  
